# Notebook 8 — Which change from M2 to M4 actually matters? (single-change ablations, MOSEI + MOSI)

**Question.** M4 differs from M2 (our adaptation of the base paper's RMSR, Eq. 10–14) in **seven** ways. Earlier notebooks
compared the whole packages only. This notebook changes **one component at a time** and measures what each one does.
It is a controlled validation experiment, **not** a novelty experiment.

**Words used below**
- *Ablation*: add, remove or swap one part of a model, keep everything else identical, and measure the change.
- *Effect of a component* = (macro-F1 of the model **with** it) − (macro-F1 of the model **without** it), in percentage points (pp).
  Positive = the component helps.
- *Modality combination*: which inputs are present at test time: T, A, V, A+V, T+A, T+V, T+A+V (T = text, A = audio, V = video).

**What differs between M2 and M4**
| Component | M2 | M4 |
|---|---|---|
| Eq. 15 alignment loss | missing (our earlier omission) | not used |
| Gate masking (missing input gets weight 0) | no | yes |
| Modality dropout during training | none | 0.3 |
| Gate formula | Eq. 11–12: sigmoid, then softmax | Linear(384→3), then softmax |
| Attention | degenerate: W_v·z (softmax over one item = 1) | token-level multi-head attention + residual |
| Input projection | Linear | Linear + LayerNorm |
| Classifier | 1 layer | 2 layers |

**Models.** M2, M3, M4 are **reused** (not retrained) if the checks in each dataset section pass.
| ID | Configuration | One change? | New trainings per dataset |
|---|---|---|---|
| M2 | reference (reused) | — | 0 |
| M3 | M4 without modality dropout (reused) | yes | 0 |
| M4 | reference (reused) | — | 0 |
| A0 | M2 + Eq. 15, λ1 = 0.3, **published form** (all samples, both pairs) | yes | 3 |
| A1 | M2 + gate masking. M2 never sees a missing input in training, so masking changes nothing there: A1 = M2's weights, masking applied at test time (checked below) | yes | 0 |
| A2 | M2 + modality dropout 0.3 | yes | 3 |
| A3 | M2 + masking + dropout | **no — two changes, labelled** | 3 |
| A4 | M2 + M4's 2-layer classifier | yes | 3 |
| B1 | M4 without gate masking | yes | 3 |
| B2 | M4 with M2's Eq. 11–12 gate (masked) | yes | 3 |
| B3 | M4 with M2-style pooled attention: h_m = W_v(mean of p_m) + mean of p_m | yes | 3 |
| B4 | M4 with the 1-layer classifier | yes | 3 |
| B5 | M4 without LayerNorm | yes | 3 |
| B6 | M4 + Eq. 15 on **available pairs only** (λ1 = 0.3) | yes | 3 |

**Eq. 15** (base paper): L_align = Σ_i Σ_{m∈{a,v}} (1 − cos(z_t^i, z_m^i)), added to the loss with λ1 = 0.3 (Eq. 21; the paper's best value,
not tuned here). Implementation: the per-sample sum over audio and video, averaged over the batch, so it has the same scale as the
(batch-averaged) cross-entropy. z_m = the pooled modality projection (Eq. 10). The paper's other loss terms (λ2, λ3) are not part
of this comparison.
- **A0** (published form): every sample, both pairs. M2 is trained without dropout, so every pair is real here.
- **B6** (M4 trains with missing inputs): a pair (text, m) counts only if **both** are present. Samples with no valid pair add 0.
  The sum is still divided by the batch size, so λ1 means the same as in A0. Missing inputs never get pulled towards anything.

**Comparisons (13).** Each row is "with − without".
| # | Component | with | without |
|---|---|---|---|
| C01 | Eq. 15 (published) on M2 | A0 | M2 |
| C02 | Gate masking on M2, no dropout (test-time only) | A1 | M2 |
| C03 | Modality dropout on M2 | A2 | M2 |
| C04 | Masking + dropout on M2 (two changes) | A3 | M2 |
| C05 | Gate masking on M2 **when dropout is used** | A3 | A2 |
| C06 | 2-layer classifier on M2 | A4 | M2 |
| C07 | Modality dropout on M3 | M4 | M3 |
| C08 | Gate masking in M4 | M4 | B1 |
| C09 | Linear gate vs Eq. 11–12 gate | M4 | B2 |
| C10 | Token-level attention vs pooled attention | M4 | B3 |
| C11 | 2-layer vs 1-layer classifier | M4 | B4 |
| C12 | LayerNorm in the projection | M4 | B5 |
| C13 | Eq. 15 (available pairs) on M4 | B6 | M4 |

**Decision rules — fixed before any result is seen**
- Two **primary measures** per comparison: (P1) test macro-F1 on T+A+V; (P2) test macro-F1 averaged over all 7 combinations.
- Interval and p-value: seed + video bootstrap, 10,000 draws (resample test videos and seeds), two-sided.
- **Holm correction** over 13 × 2 = **26** primary tests per dataset. (Holm: sort the p-values; multiply the smallest by 26, the next by 25, …;
  this keeps the chance of *any* false positive among the 26 at or below 5 %.)
- Verdict per test: **HELPS** = effect ≥ +1 pp and Holm p < .05; **HURTS** = effect ≤ −1 pp and Holm p < .05;
  *significant but below 1 pp*; *negligible* = 95 % interval inside ±1 pp; otherwise *no detectable effect* (not the same as "no effect").
- MOSEI is the primary dataset. Its test set has been used in 7 earlier notebooks, so a MOSEI result alone is weak; MOSI checks it.
- **Replication on MOSI** (only for MOSEI HELPS/HURTS): **replicated** = same sign, |MOSI effect| ≥ 1 pp, and MOSI p < .05 after Holm
  across the replication tests attempted; **same direction, not confirmed** = same sign otherwise; **opposite direction** = sign flips.
  Same direction alone is **never** called replication. Effect sizes and 95 % intervals are reported for both datasets.
- Secondary: the 13 × 7 per-combination effects, Holm over 91 tests per dataset. Validation effects are shown for reference only.
- Note: C02 on T+A+V is exactly 0 by construction (A1 = M2 when nothing is missing). It stays in the 26 as fixed in advance.

**Controls.** Same seeds (42/43/44), learning rate, batch size, epochs, early stopping (validation macro-F1, all inputs), class
weights, gradient clipping, D = 128, **4 CPU threads** — all read from the v2 checkpoints. Parameter counts are printed. The code
check below shows, for every variant, exactly which parameters were added/removed and that all shared parameters start from
**identical** initial values as the reference.

**Reuse rule (per dataset).** The M2/M3/M4 checkpoints are reused only if (a) their recorded settings equal this notebook's, and
(b) re-training M2 seed 42 here gives **bit-identical** weights (this proves the Colab environment and training code are the same).
Otherwise M2/M3/M4 are re-trained here (≈ +25 min on MOSEI), so every comparison is made inside one environment.

**Limits stated in advance.** 3 seeds only. MOSI is small (686 test clips, 31 test videos, 30 neutral) — wide intervals and
"not confirmed" are likely, honest outcomes. A component that does not help on these features may still help elsewhere.

**Runtime: CPU** (Runtime → Change runtime type → CPU, High-RAM). CPU on purpose: the reference models were trained on CPU with 4
threads, and a GPU would give different numbers and break the reuse. MOSI runs first (≈ 10 min, a quick end-to-end check), then
MOSEI (≈ 1.5 h). A projection is printed before and during training. Everything is saved as it finishes; after a disconnect,
**Run all** continues where it stopped. Writes only to `v8_ablations/`; earlier folders are read only.

In [1]:
import os, sys, time, json, random, pickle, platform, gc
IN_COLAB = "google.colab" in sys.modules
TEST_MODE = os.environ.get("EMOFAR_TEST") == "1"
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import sklearn
DEVICE = "cpu"

Mounted at /content/drive


## 1. Settings (training settings copied from the v2 checkpoints)

In [2]:
BASE_OUT = "/content/drive/MyDrive/emofar_results"
PATHS = {"MOSI": "/content/drive/MyDrive/datasets/CMU-MOSI/Processed/aligned_50.pkl",
         "MOSEI": "/content/drive/MyDrive/datasets/CMU-MOSEI/aligned_50.pkl"}
REF_DIR = {"MOSEI": f"{BASE_OUT}/mosei_v2_cpu", "MOSI": f"{BASE_OUT}/mosi_v7_confirmation"}     # read only
RUN = {"MOSI": True, "MOSEI": True}
if TEST_MODE:
    BASE_OUT = os.environ["EMOFAR_OUT"]
    PATHS = {"MOSI": os.environ["EMOFAR_MOSI"], "MOSEI": os.environ["EMOFAR_MOSEI"]}
    REF_DIR = {"MOSEI": os.environ["EMOFAR_REF_MOSEI"], "MOSI": os.environ["EMOFAR_REF_MOSI"]}
OUT_ROOT = f"{BASE_OUT}/v8_ablations"
OUT = {ds: f"{OUT_ROOT}/{ds.lower()}" for ds in PATHS}
for ds in OUT:
    os.makedirs(f"{OUT[ds]}/checkpoints", exist_ok=True); os.makedirs(f"{OUT[ds]}/preds", exist_ok=True)

v2cfg = json.load(open(f"{REF_DIR['MOSEI']}/checkpoints/M4_seed42_log.json"))["train_config"]
D_MODEL, EPOCHS, PATIENCE, BATCH_SIZE, LR, MOD_DROPOUT = (v2cfg[k] for k in ["D_MODEL", "EPOCHS", "PATIENCE", "BATCH_SIZE", "LR", "MOD_DROPOUT"])
SEEDS = [42, 43, 44]
THREADS = 4 if not TEST_MODE else int(os.environ.get("EMOFAR_THREADS", 2)); torch.set_num_threads(THREADS)
LAMBDA1 = 0.3                       # Eq. 21, base paper's value; not tuned
N_BOOT = 10000 if not TEST_MODE else 2000
MARGIN_PP, ALPHA = 1.0, 0.05
TRAIN_KEYS = ["D_MODEL", "EPOCHS", "PATIENCE", "BATCH_SIZE", "LR", "MOD_DROPOUT", "DATA_PATH"]
CONFIG = {ds: dict(D_MODEL=D_MODEL, EPOCHS=EPOCHS, PATIENCE=PATIENCE, BATCH_SIZE=BATCH_SIZE, LR=LR, MOD_DROPOUT=MOD_DROPOUT,
                   DATA_PATH=PATHS[ds], SEEDS=SEEDS, threads=THREADS) for ds in PATHS}

def cpu_name():
    try:
        return [l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")][0]
    except Exception:
        return platform.processor()

ENV = {"python": platform.python_version(), "torch": torch.__version__, "numpy": np.__version__, "sklearn": sklearn.__version__,
       "cpu": cpu_name(), "torch_threads": torch.get_num_threads(), "os": platform.platform(),
       "started_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime())}
print(json.dumps(ENV, indent=1))
print("Training settings:", {k: CONFIG["MOSEI"][k] for k in ["D_MODEL", "EPOCHS", "PATIENCE", "BATCH_SIZE", "LR", "MOD_DROPOUT", "threads"]},
      "| seeds", SEEDS, "| lambda1", LAMBDA1)
json.dump({"environment": ENV, "config": CONFIG, "lambda1": LAMBDA1, "n_boot": N_BOOT, "margin_pp": MARGIN_PP},
          open(f"{OUT_ROOT}/run_info_{time.strftime('%Y%m%d_%H%M%S', time.gmtime())}.json", "w"), indent=2)
LABELS = ["negative", "neutral", "positive"]
MODS = ["text", "audio", "vision"]
CONDS = ["T", "A", "V", "A+V", "T+A", "T+V", "T+A+V"]
ROW = {"T": (1, 0, 0), "A": (0, 1, 0), "V": (0, 0, 1), "A+V": (0, 1, 1), "T+A": (1, 1, 0), "T+V": (1, 0, 1), "T+A+V": (1, 1, 1)}

{
 "python": "3.13.18",
 "torch": "2.14.1+cpu",
 "numpy": "2.4.6",
 "sklearn": "1.6.1",
 "cpu": "Intel(R) Xeon(R) CPU @ 2.20GHz",
 "torch_threads": 4,
 "os": "Linux-6.6.159+-x86_64-with-glibc2.39",
 "started_utc": "2026-11-24 15:19:59"
}
Training settings: {'D_MODEL': 128, 'EPOCHS': 30, 'PATIENCE': 6, 'BATCH_SIZE': 64, 'LR': 0.0005, 'MOD_DROPOUT': 0.3, 'threads': 4} | seeds [42, 43, 44] | lambda1 0.3


## 2. Data loading (v2 preprocessing, unchanged; wrapped in a function so MOSI and MOSEI can run one after the other)

In [3]:
DIMS = None
def load_data(DATA_PATH, RUN_DIR):
    global DIMS
    t0 = time.time()
    with open(DATA_PATH, "rb") as f:
        raw = pickle.load(f)
    print(f"Loaded in {time.time() - t0:.0f}s")

    def prep(split):
        d = raw[split]
        mask = d["text_bert"][:, 1, :].astype(bool)                        # (N, 50) real-word steps
        feats = {"text": d["text"].astype(np.float32),
                 "audio": np.nan_to_num(d["audio"].astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0),
                 "vision": np.nan_to_num(d["vision"].astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)}
        score = d["regression_labels"].astype(np.float32)
        y = np.where(score < 0, 0, np.where(score == 0, 1, 2))
        return {"feats": feats, "mask": mask, "y": y, "score": score, "raw_text": d["raw_text"], "id": d["id"]}

    data = {s: prep(s) for s in ["train", "valid", "test"]}
    del raw

    for m in ["audio", "vision"]:                                           # standardise with train statistics
        tr = data["train"]["feats"][m][data["train"]["mask"]]
        mu, sd = tr.mean(0), tr.std(0) + 1e-6
        for s in data:
            x = (data[s]["feats"][m] - mu) / sd
            data[s]["feats"][m] = np.clip(x, -10, 10) * data[s]["mask"][..., None]

    DIMS = {m: data["train"]["feats"][m].shape[-1] for m in MODS}
    print("Feature sizes:", DIMS)
    for s in data:
        counts = np.bincount(data[s]["y"], minlength=3)
        print(f"{s:5s}: {len(data[s]['y']):6,} clips | " + " / ".join(f"{LABELS[i]} {counts[i]:,}" for i in range(3)))
    np.save(f"{RUN_DIR}/test_labels.npy", data["test"]["y"]); np.save(f"{RUN_DIR}/valid_labels.npy", data["valid"]["y"])
    np.save(f"{RUN_DIR}/test_scores.npy", data["test"]["score"])
    pd.Series(data["test"]["id"]).to_csv(f"{RUN_DIR}/test_ids.csv", index=False, header=["id"])
    return data

## 3. Reference models (v2 code, unchanged) and training functions (v2, unchanged)

In [4]:
def masked_mean(x, mask):
    m = mask.unsqueeze(-1).float()
    return (x * m).sum(1) / m.sum(1).clamp(min=1)

class TextOnly(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(DIMS["text"], D_MODEL), nn.ReLU(), nn.Dropout(0.2), nn.Linear(D_MODEL, 3))
    def forward(self, x, mask, avail, return_gate=False):
        out = self.net(masked_mean(x["text"], mask))
        return (out, None) if return_gate else out

class PaperRMSR(nn.Module):
    """Equation-level adaptation of EMO-FAR's RMSR (Eq. 10-14)."""
    def __init__(self):
        super().__init__()
        self.proj = nn.ModuleDict({m: nn.Linear(DIMS[m], D_MODEL) for m in MODS})          # Eq. 10
        self.u = nn.ParameterDict({m: nn.Parameter(torch.randn(D_MODEL) * 0.05) for m in MODS})  # Eq. 11
        self.Wq, self.Wk, self.Wv = (nn.Linear(D_MODEL, D_MODEL) for _ in range(3))
        self.cls = nn.Sequential(nn.Dropout(0.2), nn.Linear(D_MODEL, 3))
    def forward(self, x, mask, avail, return_gate=False):
        z = {m: self.proj[m](masked_mean(x[m], mask)) for m in MODS}
        q = torch.stack([torch.sigmoid((z[m] * self.u[m]).sum(-1)) for m in MODS], -1)   # Eq. 11
        gamma = q.softmax(-1)                                                             # Eq. 12
        att = {}
        for m in ["audio", "vision"]:                                                     # Eq. 13
            score = (self.Wq(z["text"]) * self.Wk(z[m])).sum(-1, keepdim=True) / D_MODEL ** 0.5
            att[m] = score.softmax(-1) * self.Wv(z[m])
        h = gamma[:, :1] * z["text"] + gamma[:, 1:2] * att["audio"] + gamma[:, 2:] * att["vision"]   # Eq. 14
        out = self.cls(h)
        return (out, gamma) if return_gate else out

class FixedFusion(nn.Module):
    """Token-level cross-attention + masked gating."""
    def __init__(self):
        super().__init__()
        self.proj = nn.ModuleDict({m: nn.Sequential(nn.Linear(DIMS[m], D_MODEL), nn.LayerNorm(D_MODEL)) for m in MODS})
        self.xattn = nn.ModuleDict({m: nn.MultiheadAttention(D_MODEL, 4, batch_first=True, dropout=0.1)
                                    for m in ["audio", "vision"]})
        self.gate = nn.Linear(3 * D_MODEL, 3)
        self.cls = nn.Sequential(nn.Dropout(0.2), nn.Linear(D_MODEL, D_MODEL), nn.ReLU(), nn.Linear(D_MODEL, 3))
    def forward(self, x, mask, avail, return_gate=False):
        p = {m: self.proj[m](x[m]) * avail[:, i, None, None] for i, m in enumerate(MODS)}
        h = {"text": masked_mean(p["text"], mask)}
        for m in ["audio", "vision"]:
            attended, _ = self.xattn[m](p["text"], p[m], p[m], key_padding_mask=~mask)
            h[m] = masked_mean(attended + p[m], mask)
        logits = self.gate(torch.cat([h[m] for m in MODS], -1))
        logits = logits.masked_fill(avail == 0, float("-inf"))
        w = logits.softmax(-1)
        fused = sum(w[:, i:i + 1] * h[m] for i, m in enumerate(MODS))
        out = self.cls(fused)
        return (out, w) if return_gate else out

MODELS = {"M1 Text-only": (TextOnly, 0.0),
          "M2 EMO-FAR RMSR (as written)": (PaperRMSR, 0.0),
          "M3 Fixed fusion": (FixedFusion, 0.0),
          "M4 Fixed fusion + modality dropout": (FixedFusion, MOD_DROPOUT)}
SHORT = {name: name.split()[0] for name in MODELS}                     # "M1" ... "M4"

from sklearn.metrics import f1_score, accuracy_score

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def sample_avail(n, rates, rng):
    """rates = (p_text, p_audio, p_vision): chance each modality is hidden. At least one stays."""
    avail = (rng.rand(n, 3) >= np.array(rates)).astype(np.float32)
    none_left = avail.sum(1) == 0
    avail[none_left, rng.randint(0, 3, none_left.sum())] = 1
    return avail

def to_batch(d, idx, avail):
    x = {m: torch.from_numpy(d["feats"][m][idx]).to(DEVICE) for m in MODS}
    for i, m in enumerate(MODS):
        x[m] = x[m] * torch.from_numpy(avail[:, i]).to(DEVICE)[:, None, None]
    return x, torch.from_numpy(d["mask"][idx]).to(DEVICE), torch.from_numpy(avail).to(DEVICE)

@torch.no_grad()
def predict(model, d, rates=(0, 0, 0), seed=0):
    model.eval(); rng = np.random.RandomState(seed); out = []
    for i in range(0, len(d["y"]), 256):
        idx = np.arange(i, min(i + 256, len(d["y"])))
        x, mask, avail = to_batch(d, idx, sample_avail(len(idx), rates, rng))
        out.append(model(x, mask, avail).softmax(-1).cpu().numpy())
    return np.concatenate(out)

def ece_score(probs, y, n_bins=15):
    conf, pred = probs.max(1), probs.argmax(1); ece = 0.0
    for lo, hi in zip(np.linspace(0, 1, n_bins + 1)[:-1], np.linspace(0, 1, n_bins + 1)[1:]):
        sel = (conf > lo) & (conf <= hi)
        if sel.any():
            ece += sel.mean() * abs((pred[sel] == y[sel]).mean() - conf[sel].mean())
    return float(ece)

def evaluate(probs, d):
    y, pred = d["y"], probs.argmax(1)
    nz = d["score"] != 0
    acc2 = ((probs[nz, 2] > probs[nz, 0]) == (d["score"][nz] > 0)).mean()
    return {"accuracy": accuracy_score(y, pred), "macro_f1": f1_score(y, pred, average="macro"),
            "acc2": float(acc2), "ece": ece_score(probs, y)}

def train(name, seed):
    cls, drop = MODELS[name]
    set_seed(seed)
    model = cls().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    counts = np.bincount(data["train"]["y"], minlength=3)
    class_w = torch.tensor(counts.sum() / (3 * counts), dtype=torch.float, device=DEVICE)
    rng = np.random.RandomState(seed); tr = data["train"]
    best, best_state, bad = -1, None, 0
    for ep in range(EPOCHS):
        model.train()
        order = rng.permutation(len(tr["y"]))
        for i in range(0, len(order), BATCH_SIZE):
            idx = order[i:i + BATCH_SIZE]
            x, mask, avail = to_batch(tr, idx, sample_avail(len(idx), (drop, drop, drop), rng))
            loss = F.cross_entropy(model(x, mask, avail), torch.from_numpy(tr["y"][idx]).to(DEVICE), weight=class_w)
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        val = evaluate(predict(model, data["valid"]), data["valid"])["macro_f1"]
        if val > best:
            best, bad = val, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    print(f"  {name:38s} seed {seed}: best valid macro-F1 = {best:.4f}  (stopped after epoch {ep + 1})")
    return model, float(best), ep + 1

def save_atomic(save_fn, path):
    """Write to a temporary file first, then rename: a disconnect never leaves a half-written file."""
    tmp = path + ".tmp"
    save_fn(tmp)
    os.replace(tmp, path)

## 4. The variants — each changes one component of M2 or M4
Each variant is a subclass of the v2 class. With its switch in the default position it **is** the v2 model (checked in Section 6).
Replaced parts are created **after** the original ones, so all shared parameters start from the same values as the reference.

In [5]:
class RMSRVar(PaperRMSR):
    """M2 with optional gate masking (A1, A3) or M4's 2-layer classifier (A4). Can return z for Eq. 15 (A0)."""
    def __init__(self, gate_mask=False, two_layer_cls=False):
        super().__init__()
        self.gate_mask = gate_mask
        if two_layer_cls:
            self.cls = nn.Sequential(nn.Dropout(0.2), nn.Linear(D_MODEL, D_MODEL), nn.ReLU(), nn.Linear(D_MODEL, 3))
    def forward(self, x, mask, avail, return_gate=False, return_z=False):
        z = {m: self.proj[m](masked_mean(x[m], mask)) for m in MODS}                         # Eq. 10
        q = torch.stack([torch.sigmoid((z[m] * self.u[m]).sum(-1)) for m in MODS], -1)       # Eq. 11
        if self.gate_mask:
            q = q.masked_fill(avail == 0, float("-inf"))                                      # missing input -> weight 0
        gamma = q.softmax(-1)                                                                 # Eq. 12
        att = {}
        for m in ["audio", "vision"]:                                                         # Eq. 13
            score = (self.Wq(z["text"]) * self.Wk(z[m])).sum(-1, keepdim=True) / D_MODEL ** 0.5
            att[m] = score.softmax(-1) * self.Wv(z[m])
        h = gamma[:, :1] * z["text"] + gamma[:, 1:2] * att["audio"] + gamma[:, 2:] * att["vision"]   # Eq. 14
        out = self.cls(h)
        if return_z:
            return out, z
        return (out, gamma) if return_gate else out

class FFVar(FixedFusion):
    """M4 with one component switched: gate masking (B1), Eq. 11-12 gate (B2), pooled attention (B3),
    1-layer classifier (B4), no LayerNorm (B5). Can return z for Eq. 15 (B6)."""
    def __init__(self, gate_mask=True, gate="linear", attention="token", classifier="two_layer", layernorm=True):
        super().__init__()
        self.gate_mask, self.gate_type, self.attention = gate_mask, gate, attention
        if not layernorm:                       # LayerNorm has no random initial values: the Linear weights stay identical
            for m in MODS:
                self.proj[m][1] = nn.Identity()
        if gate == "rmsr":                      # Eq. 11-12 on the per-modality representations h_m
            del self.gate
            self.u = nn.ParameterDict({m: nn.Parameter(torch.randn(D_MODEL) * 0.05) for m in MODS})
        if attention == "pooled":               # M2-style: W_v applied to the pooled projection, plus M4's residual
            del self.xattn
            self.wv = nn.ModuleDict({m: nn.Linear(D_MODEL, D_MODEL) for m in ["audio", "vision"]})
        if classifier == "one_layer":
            self.cls = nn.Sequential(nn.Dropout(0.2), nn.Linear(D_MODEL, 3))
    def forward(self, x, mask, avail, return_gate=False, return_z=False):
        p = {m: self.proj[m](x[m]) * avail[:, i, None, None] for i, m in enumerate(MODS)}
        h = {"text": masked_mean(p["text"], mask)}
        for m in ["audio", "vision"]:
            if self.attention == "token":
                attended, _ = self.xattn[m](p["text"], p[m], p[m], key_padding_mask=~mask)
                h[m] = masked_mean(attended + p[m], mask)
            else:
                pooled = masked_mean(p[m], mask)
                h[m] = self.wv[m](pooled) + pooled
        if self.gate_type == "linear":
            logits = self.gate(torch.cat([h[m] for m in MODS], -1))
        else:
            logits = torch.stack([torch.sigmoid((h[m] * self.u[m]).sum(-1)) for m in MODS], -1)
        if self.gate_mask:
            logits = logits.masked_fill(avail == 0, float("-inf"))
        w = logits.softmax(-1)
        fused = sum(w[:, i:i + 1] * h[m] for i, m in enumerate(MODS))
        out = self.cls(fused)
        if return_z:
            return out, {m: masked_mean(p[m], mask) for m in MODS}
        return (out, w) if return_gate else out

SPECS = {
    "M2": dict(family="ref_rmsr", arch={}, drop=0.0, eq15=None),
    "M3": dict(family="ref_ff", arch={}, drop=0.0, eq15=None),
    "M4": dict(family="ref_ff", arch={}, drop=MOD_DROPOUT, eq15=None),
    "A0": dict(family="rmsr", arch={}, drop=0.0, eq15="published"),
    "A1": dict(family="rmsr", arch={"gate_mask": True}, drop=0.0, eq15=None, weights_from="M2"),
    "A2": dict(family="rmsr", arch={}, drop=MOD_DROPOUT, eq15=None),
    "A3": dict(family="rmsr", arch={"gate_mask": True}, drop=MOD_DROPOUT, eq15=None),
    "A4": dict(family="rmsr", arch={"two_layer_cls": True}, drop=0.0, eq15=None),
    "B1": dict(family="ff", arch={"gate_mask": False}, drop=MOD_DROPOUT, eq15=None),
    "B2": dict(family="ff", arch={"gate": "rmsr"}, drop=MOD_DROPOUT, eq15=None),
    "B3": dict(family="ff", arch={"attention": "pooled"}, drop=MOD_DROPOUT, eq15=None),
    "B4": dict(family="ff", arch={"classifier": "one_layer"}, drop=MOD_DROPOUT, eq15=None),
    "B5": dict(family="ff", arch={"layernorm": False}, drop=MOD_DROPOUT, eq15=None),
    "B6": dict(family="ff", arch={}, drop=MOD_DROPOUT, eq15="valid_pairs"),
}
REF_OF = {v: ("M2" if SPECS[v]["family"] == "rmsr" else "M4") for v in SPECS if not v.startswith("M")}
REFS = ["M2", "M3", "M4"]
TRAINED = ["A0", "A2", "A3", "A4", "B1", "B2", "B3", "B4", "B5", "B6"]
COMPARISONS = [("C01", "Eq. 15 (published) on M2", "A0", "M2"),
               ("C02", "Gate masking on M2, no dropout (test-time only)", "A1", "M2"),
               ("C03", "Modality dropout on M2", "A2", "M2"),
               ("C04", "Masking + dropout on M2 (TWO changes)", "A3", "M2"),
               ("C05", "Gate masking on M2 when dropout is used", "A3", "A2"),
               ("C06", "2-layer classifier on M2", "A4", "M2"),
               ("C07", "Modality dropout on M3", "M4", "M3"),
               ("C08", "Gate masking in M4", "M4", "B1"),
               ("C09", "Linear gate vs Eq. 11-12 gate", "M4", "B2"),
               ("C10", "Token attention vs pooled attention", "M4", "B3"),
               ("C11", "2-layer vs 1-layer classifier", "M4", "B4"),
               ("C12", "LayerNorm in the projection", "M4", "B5"),
               ("C13", "Eq. 15 (available pairs) on M4", "B6", "M4")]

def build(vid):
    sp = SPECS[vid]
    if sp["family"] == "ref_rmsr":
        return PaperRMSR()
    if sp["family"] == "ref_ff":
        return FixedFusion()
    return (RMSRVar if sp["family"] == "rmsr" else FFVar)(**sp["arch"])

def align_loss(z, avail, valid_only):
    """Eq. 15: sum over audio and video of (1 - cos(z_text, z_m)), divided by the batch size (same scale as the cross-entropy).
    valid_only=False -> published form, every sample and pair. valid_only=True -> a pair counts only if text and that modality
    are both present; a sample with no valid pair adds 0. Returns (loss, number of samples with no valid pair)."""
    B = avail.shape[0]; total = z["text"].new_zeros(()); no_pair = torch.ones(B, dtype=torch.bool)
    for i, m in [(1, "audio"), (2, "vision")]:
        ok = ((avail[:, 0] > 0) & (avail[:, i] > 0)) if valid_only else torch.ones(B, dtype=torch.bool)
        no_pair &= ~ok
        if ok.any():
            total = total + (1 - F.cosine_similarity(z["text"][ok], z[m][ok], dim=-1)).sum()
    return total / B, int(no_pair.sum())

def train_variant(vid, seed):
    """v2's train(), with two additions: the model comes from SPECS, and Eq. 15 is added to the loss when the spec asks for it.
    For a spec without Eq. 15 the operations are exactly those of train() (checked bit-for-bit in Section 6)."""
    sp = SPECS[vid]; drop, eq15 = sp["drop"], sp["eq15"]
    set_seed(seed)
    model = build(vid).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    counts = np.bincount(data["train"]["y"], minlength=3)
    class_w = torch.tensor(counts.sum() / (3 * counts), dtype=torch.float, device=DEVICE)
    rng = np.random.RandomState(seed); tr = data["train"]
    best, best_state, bad, hist = -1, None, 0, []
    for ep in range(EPOCHS):
        model.train()
        order = rng.permutation(len(tr["y"]))
        s_ce, s_al, n_b, n_nopair = 0.0, 0.0, 0, 0
        for i in range(0, len(order), BATCH_SIZE):
            idx = order[i:i + BATCH_SIZE]
            x, mask, avail = to_batch(tr, idx, sample_avail(len(idx), (drop, drop, drop), rng))
            yb = torch.from_numpy(tr["y"][idx]).to(DEVICE)
            if eq15 is None:
                loss = F.cross_entropy(model(x, mask, avail), yb, weight=class_w); ce = loss
            else:
                out, z = model(x, mask, avail, return_z=True)
                ce = F.cross_entropy(out, yb, weight=class_w)
                al, nn_ = align_loss(z, avail, valid_only=(eq15 == "valid_pairs"))
                loss = ce + LAMBDA1 * al
                s_al += al.item(); n_nopair += nn_
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            s_ce += ce.item(); n_b += 1
        val = evaluate(predict(model, data["valid"]), data["valid"])["macro_f1"]
        hist.append({"epoch": ep + 1, "ce": s_ce / n_b, "align": (s_al / n_b) if eq15 else None,
                     "frac_no_valid_pair": (n_nopair / len(order)) if eq15 else None, "valid_macro_f1": float(val)})
        if val > best:
            best, bad = val, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    return model, float(best), ep + 1, hist

@torch.no_grad()
def predict_cond(model, d, cond):
    """Same as v2's predict(), but with a fixed modality combination for every clip."""
    model.eval(); out = []; row = np.array(ROW[cond], dtype=np.float32)
    for i in range(0, len(d["y"]), 256):
        idx = np.arange(i, min(i + 256, len(d["y"])))
        x, mask, avail = to_batch(d, idx, np.tile(row, (len(idx), 1)))
        out.append(model(x, mask, avail).softmax(-1).cpu().numpy())
    return np.concatenate(out)

def n_params(model):
    return sum(p.numel() for p in model.parameters())

## 5. Per-dataset pipeline: code check → reference check → train variants → predictions for all 7 combinations

In [6]:
def code_check():
    """(1) Each variant class with default switches equals the v2 class: same parameters, same initial values, same outputs.
    (2) For every variant: which parameters were removed/added/changed, and whether all shared parameters start identical."""
    rng = np.random.RandomState(0); d = data["valid"]; idx = np.arange(min(64, len(d["y"])))
    x, mask, avail = to_batch(d, idx, sample_avail(len(idx), (0.3, 0.3, 0.3), rng))
    ok = True
    for ref_cls, var_cls in [(PaperRMSR, RMSRVar), (FixedFusion, FFVar)]:
        set_seed(0); a = ref_cls().eval(); set_seed(0); b = var_cls().eval()
        sa, sb = a.state_dict(), b.state_dict()
        same = set(sa) == set(sb) and all(torch.equal(sa[k], sb[k]) for k in sa)
        same_out = torch.equal(a(x, mask, avail), b(x, mask, avail))
        print(f"  {var_cls.__name__}() with default switches == {ref_cls.__name__}: parameters identical {same}, outputs identical {same_out}")
        ok &= same and same_out
    rows = []
    for v in ["M2", "M3", "M4"] + list(REF_OF):
        r = v if v in REFS else REF_OF[v]
        set_seed(42); ref = build(r); set_seed(42); var = build(v)
        sr, sv = ref.state_dict(), var.state_dict()
        shared = [k for k in sr if k in sv and sr[k].shape == sv[k].shape]
        changed = sorted({k.rsplit(".", 1)[0] for k in sr if k in sv and sr[k].shape != sv[k].shape})
        rows.append({"model": v, "compared_to": r, "parameters": n_params(var),
                     "removed": ", ".join(sorted({k.rsplit(".", 1)[0] for k in sr if k not in sv})) or "-",
                     "added": ", ".join(sorted({k.rsplit(".", 1)[0] for k in sv if k not in sr})) or "-",
                     "shape_changed": ", ".join(changed) or "-",
                     "shared_start_identical": all(torch.equal(sr[k], sv[k]) for k in shared),
                     "modality_dropout": SPECS[v]["drop"], "eq15": SPECS[v]["eq15"] or "-"})
    tab = pd.DataFrame(rows); print(tab.to_string(index=False))
    if not ok or not tab["shared_start_identical"].all():
        raise RuntimeError("Code check failed: a variant differs from its reference in more than the intended component.")
    return tab

def ref_paths(ds, name, seed, reuse):
    base = f"{REF_DIR[ds]}/checkpoints" if reuse else f"{OUT[ds]}/checkpoints"
    return f"{base}/{name}_seed{seed}.pt", f"{base}/{name}_seed{seed}_log.json"

def reference_check(ds):
    f = f"{OUT[ds]}/reference_check.json"
    if os.path.exists(f):
        rc = json.load(open(f))
        if rc["config"] == CONFIG[ds]:
            print(f"  Reference check already done: reuse = {rc['reuse']} ({rc['reason']})"); return rc["reuse"]
    problems = []
    for name in REFS:
        for s in SEEDS:
            ckpt, logf = ref_paths(ds, name, s, True)
            if not (os.path.exists(ckpt) and os.path.exists(logf)):
                problems.append(f"{name} seed {s}: file missing"); continue
            log = json.load(open(logf)); cfg = log.get("config", log.get("train_config"))
            diff = {k: (cfg.get(k), CONFIG[ds][k]) for k in TRAIN_KEYS if cfg.get(k) != CONFIG[ds][k]}
            if "threads" in cfg and cfg["threads"] != THREADS:
                diff["threads"] = (cfg["threads"], THREADS)
            if diff:
                problems.append(f"{name} seed {s}: different settings {diff}")
    print("  Recorded settings match:", "yes" if not problems else "NO -> " + "; ".join(problems[:3]))
    maxdiff, identical, code_same = None, False, None
    if not any("missing" in p for p in problems if p.startswith("M2 seed 42")):
        t1 = time.time(); m, _, _, _ = train_variant("M2", 42)
        stored = torch.load(ref_paths(ds, "M2", 42, True)[0], map_location="cpu"); now = m.state_dict()
        identical = set(stored) == set(now) and all(torch.equal(stored[k], now[k]) for k in stored)
        maxdiff = max(float((stored[k] - now[k]).abs().max()) for k in stored) if set(stored) == set(now) else float("inf")
        print(f"  Re-trained M2 seed 42 here ({time.time() - t1:.0f}s): bit-identical to the saved checkpoint: {identical} "
              f"(largest weight difference {maxdiff:.3g})")
        if not identical:                       # tell apart "environment changed" from "training code differs"
            m0, _, _ = train("M2 EMO-FAR RMSR (as written)", 42)
            code_same = all(torch.equal(m0.state_dict()[k], now[k]) for k in now)
            print(f"  v2's own train() gives the same weights as train_variant(): {code_same} -> "
                  + ("the training code is identical; the Colab environment differs." if code_same else "CODE DIFFERENCE - stop and report."))
            if not code_same:
                raise RuntimeError("train_variant() is not equivalent to v2's train().")
    reuse = (not problems) and identical
    reason = "settings match and M2 seed 42 reproduces bit-for-bit" if reuse else ("; ".join(problems) or "M2 seed 42 does not reproduce bit-for-bit")
    print(f"  -> {'REUSING' if reuse else 'RE-TRAINING'} M2/M3/M4 " + (f"from {REF_DIR[ds]} (read only)" if reuse else f"into {OUT[ds]}"))
    save_atomic(lambda p: json.dump({"config": CONFIG[ds], "reuse": reuse, "reason": reason, "problems": problems,
                                     "m2_seed42_identical": identical, "max_abs_weight_diff": maxdiff, "code_same_as_v2": code_same,
                                     "torch": torch.__version__, "cpu": cpu_name()}, open(p, "w"), indent=2), f)
    return reuse

def spec_json(vid):
    return json.loads(json.dumps(SPECS[vid]))

def train_or_load(ds, vid, seed):
    tag = f"{vid}_seed{seed}"
    ckpt, logf = f"{OUT[ds]}/checkpoints/{tag}.pt", f"{OUT[ds]}/checkpoints/{tag}_log.json"
    if os.path.exists(ckpt) and os.path.exists(logf):
        log = json.load(open(logf))
        assert log["config"] == CONFIG[ds] and log["spec"] == spec_json(vid), f"{ckpt} was trained with other settings"
        model = build(vid); model.load_state_dict(torch.load(ckpt, map_location="cpu")); model.eval()
        return model, log, True
    t1 = time.time(); model, best, ep, hist = train_variant(vid, seed)
    log = {"model": vid, "seed": seed, "spec": spec_json(vid), "best_valid_macro_f1": best, "stopped_epoch": ep,
           "train_seconds": round(time.time() - t1, 1), "parameters": n_params(model), "history": hist, "config": CONFIG[ds]}
    save_atomic(lambda p: torch.save(model.state_dict(), p), ckpt); save_atomic(lambda p: json.dump(log, open(p, "w"), indent=2), logf)
    return model, log, False

def save_preds(ds, vid, seed, model):
    f = f"{OUT[ds]}/preds/{vid}_seed{seed}.npz"
    if os.path.exists(f):
        return False
    arr = {f"{sp}_{c.replace('+', 'p')}": predict_cond(model, data[sp], c).astype(np.float32) for sp in ["valid", "test"] for c in CONDS}
    save_atomic(lambda p: np.savez(open(p, "wb"), **arr), f)
    return True

def fmt_min(sec):
    return f"{sec / 60:.0f} min" if sec >= 90 else f"{sec:.0f} s"

def run_dataset(ds):
    global data
    t_start = time.time()
    print(f"=== {ds} ===")
    data = load_data(PATHS[ds], OUT[ds])
    print("\n[1/4] Code check"); ptab = code_check(); ptab.to_csv(f"{OUT[ds]}/parameters.csv", index=False)
    print("\n[2/4] Reference check"); reuse = reference_check(ds)
    jobs = ([(r, s) for s in SEEDS for r in REFS] if not reuse else []) + [(v, s) for s in SEEDS for v in TRAINED]
    fam = lambda v: "rmsr" if SPECS[v]["family"] in ("rmsr", "ref_rmsr") else "ff"
    prior = {}
    for f_, names in [("rmsr", ["M2"]), ("ff", ["M3", "M4"])]:
        secs = [json.load(open(ref_paths(ds, n, s, True)[1])).get("train_seconds") for n in names for s in SEEDS
                if os.path.exists(ref_paths(ds, n, s, True)[1])]
        prior[f_] = float(np.mean([x for x in secs if x])) if any(secs) else 60.0
    todo = [j for j in jobs if not os.path.exists(f"{OUT[ds]}/checkpoints/{j[0]}_seed{j[1]}.pt")]
    est = sum(prior[fam(v)] for v, _ in todo)
    print(f"\n[3/4] Training: {len(jobs)} trainings planned, {len(todo)} still to do. "
          f"Estimate from the recorded times of the reference models: about {fmt_min(est)} (updated as it runs).")
    measured = {"rmsr": [], "ff": []}; t0 = time.time()
    for k, (v, s) in enumerate(jobs, 1):
        model, log, loaded = train_or_load(ds, v, s)
        if not loaded:
            measured[fam(v)].append(log["train_seconds"])
        save_preds(ds, v, s, model)
        left = [j for j in jobs[k:] if not os.path.exists(f"{OUT[ds]}/checkpoints/{j[0]}_seed{j[1]}.pt")]
        rate = {f_: (np.mean(measured[f_]) if measured[f_] else prior[f_]) for f_ in measured}
        eq = f" | Eq.15 term {log['history'][-1]['align']:.3f}" if log["spec"]["eq15"] else ""
        status = "loaded" if loaded else f"trained in {log['train_seconds']:.0f}s, {log['stopped_epoch']} epochs"
        print(f"  [{k}/{len(jobs)}] {v} seed {s}: {status} | valid macro-F1 {log['best_valid_macro_f1']:.4f}{eq} "
              f"| elapsed {fmt_min(time.time() - t0)} | about {fmt_min(sum(rate[fam(j[0])] for j in left))} left", flush=True)
    print("\n[4/4] Predictions for all 7 combinations (references and A1)")
    for s in SEEDS:
        for r in REFS:
            ckpt, _ = ref_paths(ds, r, s, reuse)
            m = build(r); m.load_state_dict(torch.load(ckpt, map_location="cpu")); m.eval()
            save_preds(ds, r, s, m)
            if reuse:                           # evaluation check: recomputed outputs vs the probabilities saved by the earlier notebook
                stored = np.load(f"{REF_DIR[ds]}/checkpoints/{r}_seed{s}_test_probs.npy")
                now = np.load(f"{OUT[ds]}/preds/{r}_seed{s}.npz")["test_TpApV"]
                print(f"  {r} seed {s}: all-inputs test probabilities vs the saved ones: largest difference "
                      f"{np.abs(now - stored).max():.2g}, same predicted class for {100 * (now.argmax(1) == stored.argmax(1)).mean():.2f}% of clips")
            if r == "M2":                       # A1 = M2's weights with gate masking at test time
                a1 = build("A1"); a1.load_state_dict(m.state_dict()); a1.eval()
                full = predict_cond(a1, data["test"], "T+A+V")
                assert np.array_equal(full, np.load(f"{OUT[ds]}/preds/M2_seed{s}.npz")["test_TpApV"]), "A1 != M2 on full inputs"
                save_preds(ds, "A1", s, a1)
    print("  A1 gives exactly M2's outputs when all inputs are present (checked), so masking can only matter when an input is missing.")
    json.dump({"reuse_references": reuse, "seconds": round(time.time() - t_start)}, open(f"{OUT[ds]}/DONE.json", "w"))
    print(f"\n{ds} finished in {fmt_min(time.time() - t_start)}.")

## 6. MOSI (runs first: about 10 minutes, a full end-to-end check)

In [7]:
if RUN["MOSI"]:
    run_dataset("MOSI"); del data; gc.collect()

=== MOSI ===
Loaded in 5s
Feature sizes: {'text': 768, 'audio': 5, 'vision': 20}
train:  1,284 clips | negative 552 / neutral 53 / positive 679
valid:    229 clips | negative 92 / neutral 13 / positive 124
test :    686 clips | negative 379 / neutral 30 / positive 277

[1/4] Code check
  RMSRVar() with default switches == PaperRMSR: parameters identical True, outputs identical True
  FFVar() with default switches == FixedFusion: parameters identical True, outputs identical True
model compared_to  parameters                                                                removed               added shape_changed  shared_start_identical  modality_dropout        eq15
   M2          M2      152195                                                                      -                   -             -                    True               0.0           -
   M3          M3      252806                                                                      -                   -             -     

## 7. MOSEI (about 1.5 hours; resumes after a disconnect)

In [8]:
if RUN["MOSEI"]:
    run_dataset("MOSEI"); del data; gc.collect()

=== MOSEI ===
Loaded in 54s
Feature sizes: {'text': 768, 'audio': 74, 'vision': 35}
train: 16,326 clips | negative 4,738 / neutral 3,540 / positive 8,048
valid:  1,871 clips | negative 506 / neutral 433 / positive 932
test :  4,659 clips | negative 1,350 / neutral 1,025 / positive 2,284

[1/4] Code check
  RMSRVar() with default switches == PaperRMSR: parameters identical True, outputs identical True
  FFVar() with default switches == FixedFusion: parameters identical True, outputs identical True
model compared_to  parameters                                                                removed               added shape_changed  shared_start_identical  modality_dropout        eq15
   M2          M2      162947                                                                      -                   -             -                    True               0.0           -
   M3          M3      263558                                                                      -                   -

## 8. Statistics: 26 primary tests per dataset (Holm), replication on MOSI, 91 secondary tests
This section reads only saved files, so it can be re-run on its own.

In [9]:
from sklearn.metrics import f1_score

def holm(p):
    p = np.asarray(p, float); order = np.argsort(p); m = len(p); adj = np.empty(m); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, min(1.0, (m - rank) * p[i])); adj[i] = run
    return adj

def _p(b):
    return min(1.0, 2 * (min((b <= 0).sum(), (b >= 0).sum()) + 1) / (len(b) + 1))

def f1_from_counts(cm):
    cm = cm.reshape(cm.shape[:-1] + (3, 3)); tp = np.diagonal(cm, axis1=-2, axis2=-1)
    fp = cm.sum(-2) - tp; fn = cm.sum(-1) - tp; den = 2 * tp + fp + fn
    return np.where(den > 0, 2 * tp / np.maximum(den, 1e-12), 0).mean(-1)

ALL = ["M2", "M3", "M4", "A0", "A1", "A2", "A3", "A4", "B1", "B2", "B3", "B4", "B5", "B6"]
STATS, PERSEED = {}, []
for ds in ["MOSEI", "MOSI"]:
    missing = [f"{v}_seed{s}" for v in ALL for s in SEEDS if not os.path.exists(f"{OUT[ds]}/preds/{v}_seed{s}.npz")]
    if missing:
        print(f"{ds}: {len(missing)} prediction files missing ({missing[:4]} ...) -> run Section {6 if ds == 'MOSI' else 7} first."); continue
    y_te, y_va = np.load(f"{OUT[ds]}/test_labels.npy"), np.load(f"{OUT[ds]}/valid_labels.npy")
    score_te = np.load(f"{OUT[ds]}/test_scores.npy"); nz = score_te != 0
    vids = np.array([str(i).split("$_$")[0] for i in pd.read_csv(f"{OUT[ds]}/test_ids.csv")["id"]])
    _, vidx = np.unique(vids, return_inverse=True); G = vidx.max() + 1
    _r = np.random.RandomState(2026)
    W = np.stack([np.bincount(_r.randint(0, G, G), minlength=G) for _ in range(N_BOOT)]).astype(np.float32)
    SD = np.random.RandomState(99).randint(0, len(SEEDS), (N_BOOT, len(SEEDS)))
    F0, FB, FV = {}, {}, {}
    for v in ALL:
        for s in SEEDS:
            z = np.load(f"{OUT[ds]}/preds/{v}_seed{s}.npz")
            for c in CONDS:
                pt, pv = z[f"test_{c.replace('+', 'p')}"], z[f"valid_{c.replace('+', 'p')}"]
                pred = pt.argmax(1); cnt = np.zeros((G, 9)); np.add.at(cnt, (vidx, y_te * 3 + pred), 1)
                F0[(v, c, s)] = f1_score(y_te, pred, average="macro"); FB[(v, c, s)] = f1_from_counts(W @ cnt)
                FV[(v, c, s)] = f1_score(y_va, pv.argmax(1), average="macro")
                PERSEED.append({"dataset": ds, "model": v, "seed": s, "condition": c, "test_macro_f1": F0[(v, c, s)],
                                "test_acc2_nonneutral": float(((pt[nz, 2] > pt[nz, 0]) == (score_te[nz] > 0)).mean()),
                                "valid_macro_f1": FV[(v, c, s)]})

    def effect(a, b, conds):
        per_seed = np.array([np.mean([F0[(a, c, s)] - F0[(b, c, s)] for c in conds]) for s in SEEDS])
        d = np.stack([np.mean([FB[(a, c, s)] - FB[(b, c, s)] for c in conds], 0) for s in SEEDS])
        sv = d[SD, np.arange(N_BOOT)[:, None]].mean(1)
        val = np.mean([np.mean([FV[(a, c, s)] - FV[(b, c, s)] for c in conds]) for s in SEEDS])
        return {"effect_pp": 100 * per_seed.mean(), "per_seed_pp": ", ".join(f"{100 * x:+.2f}" for x in per_seed),
                "lo_pp": 100 * np.percentile(sv, 2.5), "hi_pp": 100 * np.percentile(sv, 97.5), "p": _p(sv), "valid_effect_pp": 100 * val}

    prim, sec = [], []
    for cid, comp, a, b in COMPARISONS:
        for meas, conds in [("P1 T+A+V", ["T+A+V"]), ("P2 mean of 7", CONDS)]:
            prim.append({"dataset": ds, "id": cid, "component": comp, "with": a, "without": b, "measure": meas, **effect(a, b, conds)})
        for c in CONDS:
            sec.append({"dataset": ds, "id": cid, "component": comp, "with": a, "without": b, "condition": c, **effect(a, b, [c])})
    prim, sec = pd.DataFrame(prim), pd.DataFrame(sec)
    prim["p_holm"] = holm(prim["p"]); sec["p_holm"] = holm(sec["p"])

    def verdict(r):
        if r["p_holm"] < ALPHA and r["effect_pp"] >= MARGIN_PP: return "HELPS"
        if r["p_holm"] < ALPHA and r["effect_pp"] <= -MARGIN_PP: return "HURTS"
        if r["p_holm"] < ALPHA: return "significant, below 1 pp"
        if r["lo_pp"] > -MARGIN_PP and r["hi_pp"] < MARGIN_PP: return "negligible (CI within ±1 pp)"
        return "no detectable effect"
    prim["verdict"] = prim.apply(verdict, axis=1); sec["verdict"] = sec.apply(verdict, axis=1)
    STATS[ds] = {"primary": prim, "secondary": sec}

PERSEED = pd.DataFrame(PERSEED); PERSEED.to_csv(f"{OUT_ROOT}/per_seed_all_conditions.csv", index=False)
if "MOSEI" in STATS:
    P = STATS["MOSEI"]["primary"].copy()
    if "MOSI" in STATS:
        Q = STATS["MOSI"]["primary"].set_index(["id", "measure"])
        for col in ["effect_pp", "lo_pp", "hi_pp", "p", "p_holm", "verdict"]:
            P["MOSI_" + col] = [Q.loc[(i, m), col] for i, m in zip(P["id"], P["measure"])]
        tested = P["verdict"].isin(["HELPS", "HURTS"])
        P["MOSI_p_holm_replication"] = np.nan
        if tested.any():
            P.loc[tested, "MOSI_p_holm_replication"] = holm(P.loc[tested, "MOSI_p"])
        def repl(r):
            if r["verdict"] not in ("HELPS", "HURTS"):
                return "MOSI raw p<.05 only (exploratory)" if (r["MOSI_p"] < ALPHA and abs(r["MOSI_effect_pp"]) >= MARGIN_PP) else "-"
            if np.sign(r["MOSI_effect_pp"]) != np.sign(r["effect_pp"]): return "opposite direction"
            if abs(r["MOSI_effect_pp"]) >= MARGIN_PP and r["MOSI_p_holm_replication"] < ALPHA: return "REPLICATED"
            return "same direction, not confirmed"
        P["replication"] = P.apply(repl, axis=1)
    P.to_csv(f"{OUT_ROOT}/primary_tests.csv", index=False)
    pd.concat([STATS[d]["secondary"] for d in STATS]).to_csv(f"{OUT_ROOT}/secondary_per_condition.csv", index=False)
    if "MOSI" in STATS:
        STATS["MOSI"]["primary"].to_csv(f"{OUT_ROOT}/primary_tests_MOSI_own_holm.csv", index=False)

## 9. Results

In [10]:
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 60)
for ds in STATS:
    t = PERSEED[PERSEED.dataset == ds].pivot_table(index="model", columns="condition", values="test_macro_f1", aggfunc="mean")[CONDS]
    t["mean of 7"] = t.mean(axis=1)
    print(f"\n{ds}: test macro-F1 (mean of 3 seeds) for every model and input combination")
    print((100 * t.loc[ALL]).round(2).to_string())
print("\nParameters:"); print(pd.read_csv(f"{OUT['MOSEI' if 'MOSEI' in STATS else 'MOSI']}/parameters.csv")[["model", "compared_to", "parameters", "removed", "added", "shape_changed"]].to_string(index=False))

if "MOSEI" in STATS:
    show = P.copy()
    show["MOSEI effect [95% CI]"] = show.apply(lambda r: f"{r.effect_pp:+.2f} [{r.lo_pp:+.2f}, {r.hi_pp:+.2f}]", axis=1)
    show["Holm p"] = show["p_holm"].map(lambda x: f"{x:.3f}")
    cols = ["id", "component", "measure", "MOSEI effect [95% CI]", "Holm p", "verdict"]
    if "MOSI" in STATS:
        show["MOSI effect [95% CI]"] = show.apply(lambda r: f"{r.MOSI_effect_pp:+.2f} [{r.MOSI_lo_pp:+.2f}, {r.MOSI_hi_pp:+.2f}]", axis=1)
        cols += ["MOSI effect [95% CI]", "replication"]
    print("\nPRIMARY RESULTS — effect of each component = with − without, test macro-F1 in pp (positive = the component helps)")
    print("Holm over 26 tests on MOSEI. Replication on MOSI as defined at the top.")
    print(show[cols].to_string(index=False))
    sec = STATS["MOSEI"]["secondary"]
    grid = sec.pivot_table(index="id", columns="condition", values="effect_pp")[CONDS].round(2).astype(str)
    star = sec.pivot_table(index="id", columns="condition", values="p_holm")[CONDS] < ALPHA
    print("\nSecondary (MOSEI): effect per input combination, pp; * = Holm p < .05 over 91 tests")
    print(grid.where(~star, grid + "*").to_string())

    print("\nPLAIN SUMMARY (MOSEI primary measures; MOSI replication in brackets)")
    for cid, comp, a, b in COMPARISONS:
        rr = P[P.id == cid]
        txt = "; ".join(f"{r.measure.split()[0]} {r.verdict} ({r.effect_pp:+.2f} pp)" + (f" [MOSI: {r.replication}]" if getattr(r, "replication", "-") != "-" else "")
                        for r in rr.itertuples())
        print(f"  {cid} {comp:48s}: {txt}")
    json.dump({"primary": P.to_dict(orient="records"),
               "rules": {"holm_family": len(P), "margin_pp": MARGIN_PP, "alpha": ALPHA, "n_boot": N_BOOT, "seeds": SEEDS, "lambda1": LAMBDA1}},
              open(f"{OUT_ROOT}/VERDICTS.json", "w"), indent=2, default=str)
    print(f"\nSaved: {OUT_ROOT}/primary_tests.csv, secondary_per_condition.csv, per_seed_all_conditions.csv, VERDICTS.json")


MOSEI: test macro-F1 (mean of 3 seeds) for every model and input combination
condition      T      A      V    A+V    T+A    T+V  T+A+V  mean of 7
model                                                                
M2         61.65  31.62  34.08  36.05  61.78  62.35  62.27      49.97
M3         62.15  37.98  36.42  38.95  61.85  62.86  62.87      51.87
M4         62.67  40.02  39.45  41.76  62.81  63.57  63.45      53.39
A0         61.58  13.25  18.86  21.23  61.82  62.07  62.20      43.00
A1         61.64  34.96  35.57  36.56  61.71  62.20  62.27      50.70
A2         61.66  39.59  39.15  41.89  62.19  62.56  62.66      52.81
A3         61.68  38.90  37.59  41.26  62.02  62.14  62.61      52.32
A4         62.48  32.26  33.14  35.56  62.60  62.93  63.22      50.32
B1         61.42  38.41  39.42  40.96  62.02  62.61  62.75      52.51
B2         62.04  39.27  39.56  42.06  62.51  62.88  62.94      53.04
B3         62.81  40.39  40.29  42.70  62.94  63.48  63.22      53.69
B4         6